In [49]:
import gymnasium as gym
from gymnasium import spaces
import math
import numpy as np
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from treys import Card, Deck, Evaluator

evalu = Evaluator()

# -----------------------------------------------------------
# Monte Carlo Equity Calculator
# -----------------------------------------------------------

def simulate_preflop(h1, h2):

    your_hand = [Card.new(h1), Card.new(h2)]

    count = 0


    N = 2500  

    for i in range(N):
        deck = Deck()

        # remove hero hand from deck
        deck.cards.remove(your_hand[0])
        deck.cards.remove(your_hand[1])

        villain_hand = deck.draw(2)
        board = deck.draw(5)

        hero_score = evalu.evaluate(board, your_hand)
        villain_score = evalu.evaluate(board, villain_hand)


        if villain_score > hero_score:
            count += 1
        else:
            pass

    win_rate = count / N


    return win_rate

def simulate_preturn(h1, h2, b1, b2, b3):

    your_hand = [Card.new(h1), Card.new(h2)]
    flop = [Card.new(b1), Card.new(b2), Card.new(b3)]

    count = 0

    N = 2500

    for i in range(N):
        deck = Deck()

        # remove hero cards and flop cards
        deck.cards.remove(your_hand[0])
        deck.cards.remove(your_hand[1])
        for c in flop:
            deck.cards.remove(c)

        villain_hand = deck.draw(2)
        turn_river = deck.draw(2)
        board = flop + turn_river

        hero_score = evalu.evaluate(board, your_hand)
        villain_score = evalu.evaluate(board, villain_hand)


        if villain_score > hero_score:
            count += 1
        else:
            pass

    win_rate = count / N


    return win_rate

def simulate_preriver(h1, h2, b1, b2, b3, b4):

    your_hand = [Card.new(h1), Card.new(h2)]
    flop_turn = [
        Card.new(b1), 
        Card.new(b2), 
        Card.new(b3), 
        Card.new(b4)
    ]

    count = 0

    N = 2500

    for i in range(N):
        deck = Deck()

        deck.cards.remove(your_hand[0])
        deck.cards.remove(your_hand[1])

        for c in flop_turn:
            deck.cards.remove(c)

        villain_hand = deck.draw(2)
        river = deck.draw(1)
        board = flop_turn + river

        hero_score = evalu.evaluate(board, your_hand)
        villain_score = evalu.evaluate(board, villain_hand)


        if villain_score > hero_score:
            count += 1
        else:
            pass

    win_rate = count / N

    return win_rate

def simulate_prereveal(h1, h2, b1, b2, b3, b4, b5):

    your_hand = [Card.new(h1), Card.new(h2)]
    board = [
        Card.new(b1),
        Card.new(b2),
        Card.new(b3),
        Card.new(b4),
        Card.new(b5)
    ]

    hero_score = evalu.evaluate(board, your_hand)

    count = 0

    N = 2500

    for i in range(N):
        deck = Deck()
        deck.cards.remove(your_hand[0])
        deck.cards.remove(your_hand[1])
        for card in board:
            deck.cards.remove(card)
            
        villain_hand = deck.draw(2)

        villain_score = evalu.evaluate(board, villain_hand)

        if villain_score > hero_score:
            count += 1
        else:
            pass

    win_rate = count / N

    return win_rate


# -----------------------------------------------------------
# Environment 
# -----------------------------------------------------------
"""
Notes for Poker class:
This is the third iteration of the environment, although past environments have worked they were working on poor game mechanics,
as well as hitting a plateu on the rewards at around 0.02 (normalized) chips per game, on a set amount of the stack.
The step function's step was one betting round at any given street. On this iteration, I made the stack a random number between 
100, and 500 (since this is a reasonable range for casual play), and the step is only one action (of the hero, as well as the 
opponent's response).
I might add another observation element - that being a boolean flag of if the betting round is live or not.
Nevertheless, I've given the observation vector more elements, and made the step function more simple. I believe this will 
lead to the bot learning the overpowered strategies its supposed to a lot slower, as I will need a lot more episodes to compensate
for this increase in information the network will be handling. 
Furthermore, I've implemented some strategies like pot odds, and reward shaping to help push it past that 0.02 chips per game.
"""

class Poker(gym.Env):
    def __init__(self):
        """
        Notes for __init__ function:
        Pot, hero stack, effective stack, are normalized by using rate-based adjustment - dividing by starting stack. 
        While starting stack is normalized by using min-max rescaling - x - x_min/ x_max - x_min.
        """
        super().__init__()

        self.observation_space = spaces.Box(
            low = np.array([
                0.0, # win rate
                0.0, # starting stack (starting stacks are equal for both opp and hero)
                0.0, # pot
                0.0, # hero stack
                0.0, # effective stack
                0.0, # position
                0.0, # pot odds
                0.0, # opp agg
                0.0, # street
                0.0,  # opp last action
                0.0
            ], dtype = np.float32),
            high = np.array([
                1.0,
                1.0, 
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                5.0,
                1.0,
                6.0,
                1.0
            ], dtype = np.float32),
            dtype = np.float32
        )

        self.action_space = spaces.Discrete(7) # 0 -> fold, 1 -> check/call, 2 -> bet 33% of pot, 3 -> bet 67% of pot, 4 -> bet 100% of pot, 5 -> bet 150% of pot, 5 -> all-in
        
        self.button = None # 0 for small blind, 1 for big blind
        self.small_blind = None # will be set to 0.02x of stack
        self.big_blind = None # big blind is 2x small blind
        self.stack = None # will be randomly set within a normal range

        self.deck = None
        self.street = None # 0 --> preflop, 1 --> after flop, 2 --> after turn, 3 --> after after river
        self.board = None
        self.h1 = None
        self.h2 = None

        self.opp_stack = None
        self.opp_raises = None
        self.opp_calls = None
        self.opp_action = None
       
        self.bet_amount = None # bet amount (will be on big blind for every street except 0th - preflop)
        self.bet_history = None # will be a list of current bet and bet on the last iteration, can get call/raise amounts, and a chekc if betting is finished
       
        self.chips_to_pot = None # chips betting is symmetric so we only need one, must createa  funciton that checks if other person calls though
        self.pot = None
        
        self.hero_stack = None
        self.win_rate = None
    
        self.betting_ongoing = None # will be a boolean flag telling us if a betting round is going on, 0 for no, 1 for yes
        self.turn = None # will be used while betting_live is true, 0 for hero, 1 for opp

    def _get_obs(self):
        
        call_amount = self.bet_history[1] - self.bet_history[0]
        
        if call_amount == 0:
            pot_odds = 0
        else:
            pot_odds = call_amount / (self.pot + call_amount)

        opp_agg = min(self.opp_raises / max(self.opp_calls, 1), 5)
        return np.array([
            self.win_rate, 
            (self.stack - 100) / 400,
            self.pot / (2 * self.stack), # normalized pot
            self.hero_stack / self.stack, # normalized hero stack
            min(self.hero_stack, self.opp_stack) / self.stack, # normalized effective stack
            self.button,
            pot_odds, # pot odds (doesn't inlcude opponent last bet as it is already accounted for in the pot)
            opp_agg, 
            self.street / 3,
            self.opp_action,
            self.betting_ongoing
        ], dtype = np.float32)


    def reset(self, seed = None, options = None):
        super().reset(seed=seed)
        
        self.deck = Deck()
        self.board = []

        hero_cards = self.deck.draw(2)
        self.h1 = Card.int_to_str(hero_cards[0])
        self.h2 = Card.int_to_str(hero_cards[1])
    
        self.button = random.randint(0,1)
        self.street = 0
        self.betting_ongoing = False
        
        self.stack = random.randint(100, 500)
        self.hero_stack = self.stack
        self.opp_stack = self.stack
        self.small_blind = self.stack * 0.02
        self.big_blind = 2 * self.small_blind
        
        self.opp_raises = 0
        self.opp_calls = 0

        self.bet_amount = 0
        self.bet_history = deque([0,0],maxlen=2)
       
        self.chips_to_pot = 0
        self.pot = 0
        self.compute_win_rate()

        observation = self._get_obs()

        return observation, {}

    def step(self, action):
        terminated = False
        truncated = False

        hero_fold = self.apply_action(action)

        if hero_fold == True:
            terminated = True
            reward = self.get_reward()
            observation = self._get_obs()
            return observation, reward, terminated, truncated, {}

        move = self.done(action)

        if move == "advance":
            self.advance_street(move)

            if self.street > 3:
                termianted = True
                reward = self.get_reward()
                observation = self._get_obs()
                
                return observation, reward, terminated, truncated, {}
                
        self.compute_win_rate()

        while not terminated:
            self.whos_turn()

            if self.turn == 0:
                break

            opp_action = self.opp()

            if opp_action == 0:
                terminated = True
                reward = self.get_reward()
                observation = self._get_obs()

                return observation, reward, terminated, truncated, {}

            move = self.done(None)

            if move == "advance":
                self.advance_street(move)

                if self.street > 3:
                    terminated = True
                    reward = self.get_reward()
                    observation = self._get_obs()

                    return observation, reward, terminated, truncated, {}

                self.compute_win_rate()
                
        reward = self.reward_shaping(action)
        observation = self._get_obs()

        return observation, reward, terminated, truncated, {}
        

    def advance_street(self, move):
        """
        Notes for advance_street function:
        This function will advance the street and reset all respective variables
        This function will be called after betting is done and will take in the flag the done function gives
        """
        if move == "advance":
            self.street += 1
            self.bet_history = deque([0,0],maxlen=2)
            self.chips_to_pot = 0
            
            if self.street == 1:
                self.board.extend(self.deck.draw(3))

            elif self.street == 2:
                self.board.extend(self.deck.draw(1))

            elif self.street == 3:
                self.board.extend(self.deck.draw(1))

        
    def compute_win_rate(self):
        """ 
        Notes for compute_win_rate function:
        Don't know if I need return statement since the variable being updated is global (within the class)
        """
        if self.street == 0:
            self.win_rate = simulate_preflop(self.h1, self.h2)
            return self.win_rate
            
        elif self.street == 1:
            board = [Card.int_to_str(card) for card in self.board]
            self.win_rate = simulate_preturn(self.h1, self.h2, board[0], board[1], board[2])
            return self.win_rate
            
        elif self.street == 2:
            board = [Card.int_to_str(card) for card in self.board]
            self.win_rate = simulate_preriver(self.h1, self.h2, board[0], board[1], board[2], board[3])
            return self.win_rate
        else:
            board = [Card.int_to_str(card) for card in self.board]
            self.win_rate = simulate_prereveal(self.h1, self.h2, board[0], board[1], board[2], board[3], board[4])
            return self.win_rate

            
    def post_blinds(self):
        if self.button == 0:
            if self.street == 0:
                self.pot_mover("hero",self.small_blind)
                self.pot_mover("opp", self.big_blind)
        else:
            if self.street == 0:
                self.pot_mover("opp", self.big_blind)
                self.pot_mover("hero", self.small_blind)

            
    def first_to_act(self):
        """
        Notes for first_to_act function:
        Can we negate the while loops entirely?
        This function will return who is first to act at the beginning of each street
        This function should be called after the incremation of the street in step()
        """
        if self.button == 0:
            if self.street == 0:
                self.turn = 0
                self.betting_ongoing = True
                return "hero"
            else:
                while self.street in (1,2,3):
                    self.betting_ongoing = True
                    self.turn = 1
                    return "opp"
        else:
        # opponent has button
            if self.street == 0:
                self.betting_ongoing = True
                self.turn = 1
                return "opp"
            else:
                while self.street in (1,2,3):
                    self.betting_ongoing = True
                    self.turn = 0
                    return "hero"

                    
    def whos_turn(self):
        """
        Notes for whos_turn function:
        This function updates who's turn it is given any street, and any position of the betting round
        This function will be called in step and relies on self.turn
        """
        if self.turn is None:
            first = self.first_to_act()
            self.turn = 0 if first == "hero" else 1
            
        if not self.betting_ongoing:
            first = self.first_to_act()
            self.turn = 0 if first == "hero" else 1

        else:
            self.turn = 1 - self.turn


    def done(self, action):
        """
        Notes for done function:
        This function will update self.betting_end and will take in hero's action to do so, and return advance flag for advance_street
        This means the function will be called after the heros action AND after the opponents'
        """
        call_amount = self.bet_history[1] - self.bet_history[0]
        if call_amount == 0 or self.opp_action == 0 or action == 0:
            self.betting_ongoing = True
        else:
            self.betting_ongoing = False
            return "advance"


    def chip_mover(self, player, amount):
        """
        Notes for chip_mover function:
        This function will update stacks and pot for the respective case
        This function should be used in opponent model as well as apply_action function
        """
        if player == "hero":
            self.hero_stack -= amount
            self.pot += amount
        else:
            self.opp_stack -= amount
            self.pot += amount


    def opp(self):
        """
        Notes for opp function:
        This function will model opponent actions, and execute the respective consequence
        This function should be used in step()
        Eventually should implement stochastics where a high opponent aggression increases the probablility for raises
        """
        action = random.randint(0,6)

        if action == 0:
            return 0
            
        elif action == 1:
            call_amount = self.bet_history[1] - self.bet_history[0]
            self.chip_mover("opp", call_amount)
            
        elif action in (2, 3, 4, 5):
            call_amount = self.bet_history[1] - self.bet_history[0]

            if action == 2:
                bet_amount = call_amount + 0.33*self.pot
                self.bet_history.append(bet_amount)
                
            elif action == 3:
                bet_amount = call_amount + 0.67*self.pot
                self.bet_history.append(bet_amount)
            elif action == 4:
                bet_amount = call_amount + self.pot
                self.bet_history.append(bet_amount)
            elif action == 5:
                bet_amount = call_amount + 1.25*self.pot
                self.bet_history.append(bet_amount)
                
            self.chip_mover("opp", bet_amount)
        elif action == 6:
            self.chip_mover("opp", self.opp_stack)
            self.bet_history.append(self.opp_stack)

        return action

    def apply_action(self, action):
        """
        Notes for apply_action function:
        This function applies respective consequences of whatever action the agent chooses
        while returning if it has folded or not
        """
        terminated = False

        if action == 0:
            terminated = True
            
        elif action == 1:
            call_amount = self.bet_history[1] - self.bet_history[0]
            self.chip_mover("hero", call_amount)
            
        elif action in (2, 3, 4, 5):
            call_amount = self.bet_history[1] - self.bet_history[0]

            if action == 2:
                bet_amount = call_amount + 0.33*self.pot
                self.bet_history.append(bet_amount)
                
            elif action == 3:
                bet_amount = call_amount + 0.67*self.pot
                self.bet_history.append(bet_amount)
                
            elif action == 4:
                bet_amount = call_amount + self.pot
                self.bet_history.append(bet_amount)
                
            elif action == 5:
                bet_amount = call_amount + 1.25*self.pot
                self.bet_history.append(bet_amount)

            self.chip_mover("hero", bet_amount)
        elif action == 6:
            self.chip_mover("hero", self.hero_stack)
            self.bet_history.append(self.hero_stack)
    
        return terminated


    def showdown(self):
        """
        Notes for showdown function:
        This function just tells us if opponent wins or loses once termination occurs
        """
        opp_hand = self.deck.draw(2) 
        opp_score = evalu.evaluate(self.board, opp_hand)
        hero_score = evalu.evaluate(self.board, [Card.new(self.h1), Card.new(self.h2)])

        if opp_score < hero_score:
            
            return "loss"
            
        elif hero_score < opp_score:
            
            return "win"
            
        else:

            return None

        
    def reward_shaping(self, action):
        """
        Notes for reward_shaping function:
        This function will reinforce good strategy. as well as penalize dumb actions
        This function should be called inside the actual reward function
        """
        call_amount = self.bet_history[1] - self.bet_history[0]
        reward = 0
        
        if call_amount == 0:
            pot_odds = 0
        else:
            pot_odds = call_amount / (self.pot + call_amount)

            
# pot odds strategies        
        if pot_odds < self.win_rate and action == 1:
            reward = 5
        elif pot_odds > self.win_rate and action == 1:
            reward = -5


# folding scenarios
        if self.win_rate >= 0.75 and action == 0:
            reward = -10

        elif self.win_rate >= 0.75 and action != 0:
            reward = 10

        elif 0.5 <= self.win_rate < 0.75 and action == 0:
            reward = -5

        elif 0.5 <= self.win_rate < 0.75 and action != 0:
            reward = 5

        elif 0 <= self.win_rate <= 0.25 and action == 0:
            reward = 5

        elif 0 <= self.win_rate <= 0.25 and action != 0:
            reward = -5
            
        else:
            pass

        return reward


    def get_reward(self):
        result = self.showdown()
        
        if result == "win":
            reward = self.pot - self.stack
        elif result == None:
            reward = 0
        else:
            reward = self.stack - self.pot

        return reward 

    
    def action_mask(self):
        return np.array(
            [True, True, True, True, True, True, True],
            dtype=bool
        )



In [50]:
env = Poker()

is_ipython = 'inline' in matplotlib.get_backend() # checking if program is run in a notebook or command terminal
if is_ipython:
    from IPython import display

plt.ion()

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

Transition = namedtuple("Transition", ('state', 'action', "next_state", 'reward'))

class MemoryBank(object):

    def __init__(self, bound):
        self.memory =  deque([], maxlen = bound)
        
    def __len__(self):
        return len(self.memory)
        
    def push(self, *args):
        self.memory.append(Transition(*args))
        
    def sample(self, size):
        return random.sample(self.memory, size)


class DQN(nn.Module):

    def __init__(self, n_obs, n_actions):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(n_obs, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, n_actions)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

BATCH = 128
GAMMA = 0.99
EPS_0 = 0.9
EPS_FINAL = 0.01
EPS_DECAY = 12500
TAU = 0.005
ALPHA = 3e-4

n_actions = 7
observation, info = env.reset()
n_obs = len(observation)    

policy_net = DQN(n_obs, n_actions).to(device) 
target_net = DQN(n_obs, n_actions).to(device) 
target_net.load_state_dict(policy_net.state_dict()) 

optimizer = optim.AdamW(policy_net.parameters(), lr = ALPHA, amsgrad = True) 
memory = MemoryBank(125000)
steps = 0

def select_action(state):
    global steps
    n = random.random()
    eps_upper = EPS_FINAL + (EPS_0 - EPS_FINAL) * math.exp(-1.0 * steps / EPS_DECAY)
    steps += 1

    if n > eps_upper:
        with torch.no_grad():
            q_values = policy_net(state)

            mask = torch.tensor(env.action_mask(), device = device)
            q_values[:,~mask] = -1e9
            
            return q_values.argmax(1).view(1,1)
    else:
        legal = np.where(env.action_mask())[0]
        action = random.choice(legal)
        return torch.tensor([[action]], device = device, dtype = torch.long)

reward_episodes = []

def plot_reward(show_result = False):
    plt.figure(1)
    reward_t = torch.tensor(reward_episodes, dtype = torch.float)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel('Reward')
    plt.scatter(range(len(reward_t)), reward_t.numpy(), s=8, color='blue')
    
    if len(reward_t) >= 100:
        means = reward_t.unfold(0, 100, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())

    plt.pause(0.001)  
    if is_ipython:
        if not show_result:
            display.display(plt.gcf())
            display.clear_output(wait=True)
        else:
            display.display(plt.gcf())

def optimize_model():
    if len(memory) < BATCH:
        return
    transitions = memory.sample(BATCH)
    batch = Transition(*zip(*transitions))

    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None, batch.next_state)), device=device, dtype=torch.bool)
    non_final_next_states = torch.cat([s for s in batch.next_state if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    state_action_values = policy_net(state_batch).gather(1, action_batch)

    next_state_values = torch.zeros(BATCH, device=device)
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values

    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()

num_episodes = 20000
for episode in range(num_episodes):
    state, info = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    total_reward = 0
    
    for t in count():
        action = select_action(state)
        obs, reward, terminated, truncated, info = env.step(action.item())

        reward = reward / env.stack         
        reward = torch.tensor([reward], dtype=torch.float32, device=device)

        done = terminated or truncated

        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)

        memory.push(state, action, next_state, reward)
        state = next_state

        optimize_model()

        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key]*TAU + target_net_state_dict[key]*(1-TAU)
        target_net.load_state_dict(target_net_state_dict)
        total_reward += reward.item()
        if done:
            reward_episodes.append(total_reward)
            plot_reward()
            break
weights = torch.save(policy_net.state_dict(), "policy_net_weights.pth")
print('Complete')
plot_reward(show_result=True)
plt.ioff()
plt.show()

KeyError: 2

<Figure size 640x480 with 0 Axes>